# NINE-Music no GitHub Codespaces

Treine uma IA de música (ABC notation) do zero, em CPU, direto no Codespace.

**Fluxo:** dataset Nottingham → tokenizer BPE → Transformer tiny → geração ABC/MIDI/WAV.

> Se o Codespace acabou de abrir, espere o `postCreateCommand` terminar.

In [ ]:
import os, sys
print(sys.version)
try:
    import torch
    print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
except ImportError:
    print('torch ausente; rodando setup...')
    !bash scripts/codespace_setup.sh

## 1. Dataset + tokenizer

Baixa o Nottingham Music Database (~1.100 tunes), normaliza, transpõe para 6 tonalidades
e adiciona 300 tunes sintéticas. Depois treina o BPE (vocab 512) e gera o `music.bin`.

In [ ]:
DATA = 'nine/data/music'
if not os.path.exists(f'{DATA}/music.bin'):
    !python -m nine.music_data --out {DATA} --augment 6 --synth 300
    !python -m nine.prep_data --paths {DATA}/corpus.txt --out {DATA}/corpus_clean.txt --train_bpe --vocab 512 --bin_out {DATA}/music.bin --tok_out {DATA}/music-tok.json
else:
    print('dataset ja existe:', DATA)

for name in ('corpus.txt', 'music.bin', 'music-tok.json'):
    path = f'{DATA}/{name}'
    if os.path.exists(path):
        print(f'{name}: {os.path.getsize(path)/1024:.1f} KB')

## 2. Treino

`demo` é um teste rápido do pipeline (~poucos minutos). `cpu` é o preset padrão
(~1.7M parâmetros). Para mais qualidade, use `gpu` em uma máquina com CUDA.

Ajuste `ITERS` se quiser treinar mais:

```bash
ITERS=1800 bash scripts/train_music.sh cpu
```

In [ ]:
!bash scripts/train_music.sh cpu

## 3. Gerar e ouvir

O modelo gera ABC; o projeto converte para MIDI e WAV com síntese aditiva local.

In [ ]:
!python -m nine.music_cli --style jig --num 2 --tokens 320 --out nine/data/music/samples/notebook

from IPython.display import Audio, display
for suffix in ('_01', '_02'):
    path = f'nine/data/music/samples/notebook{suffix}.wav'
    if os.path.exists(path):
        print(path)
        display(Audio(path))

## 4. Experimente variações

Mude estilo (`reel`, `jig`, `waltz`, `slip`, `air`), tonalidade e temperatura.

In [ ]:
!python -m nine.music_cli --style reel --key Am --temp 1.0 --num 1 --tokens 360 --out nine/data/music/samples/reel_am

## 5. Download do checkpoint

O checkpoint também fica em `nine/data/music/nine-music.pt` (ignorado pelo git).
Baixe pelo VS Code (botão direito → Download) ou use o link abaixo.

In [ ]:
from IPython.display import FileLink, display
ckpt = 'nine/data/music/nine-music.pt'
print('checkpoint:', os.path.getsize(ckpt)/1024/1024, 'MB')
display(FileLink(ckpt))

## 6. Interface web

No terminal do Codespace:

```bash
python music_ui.py
```

A porta **7860** é encaminhada automaticamente.